In [1]:
##TFBS Analyzer (Ensembl Auto-Fetch + DUAL Scan + KEGG API)
#Description:
#This script extracts the exact promoter sequence using an Ensembl ID, scans for Transcription Factor Binding Sites (TFBS) utilizing JASPAR and HOCOMOCO Position Weight Matrices (PWMs), and filters false positives using a 
#Machine Learning model (Random Forest) based on genomic context. Finally, it maps the findings to oncogenic pathways and clinical diseases using the MaayanLab Enrichr API (KEGG & DisGeNET).

#INPUTS (User Configuration):
#1. TARGET_GENE_ID: Ensembl ID of the target gene.
#   2. UPSTREAM_BP / DOWNSTREAM_BP: Promoter window dimensions relative to TSS.
#   3. JASPAR_THRESHOLD: Minimum relative affinity score for PWM matching.
#   4. OUTPUT_DIR: Desired folder path to save the generated CSV files.

#OUTPUTS:
#    - Base_Results_[Gene].csv: Consolidated catalog of all viable TFBS.
#    - Oncological_Filter_[Gene].csv: Subset of TFs with direct cancer associations.
#     - Metabolic_Pathways_[Gene].csv: Subset of TFs associated with general pathways.

#Requirements:
# pip install biopython requests scikit-learn numpy pandas urllib3

import requests
import numpy as np
import pandas as pd
from Bio import motifs
from Bio.Seq import Seq
from io import StringIO
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
import re
import time
import urllib3
import os

# Disable SSL warnings for the API requests (bypasses local firewall issues)
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# USER CONFIGURATION

# Target Gene Ensembl ID
TARGET_GENE_ID = "ENSG00000091127" 

# Promoter window coordinates (base pairs)
UPSTREAM_BP = 2000    
DOWNSTREAM_BP = 500   

# Minimum affinity threshold for the primary PWM scan
JASPAR_THRESHOLD = 0.80 

# Desired output directory for the generated CSV files
OUTPUT_DIR = "./results"

# RUN THE MODULES SETUP WITH YOUR INPUTS

In [2]:
# SETUP MODULE 1: ENSEMBL API

def download_ensembl_promoter(gene_id, upstream=2000, downstream=500):
    #Fetches the exact gene coordinates and promoter sequence using Ensembl ID#
    print(f"Connecting to Ensembl API to retrieve coordinates for ID: {gene_id}...")
    
    url_lookup = f"https://rest.ensembl.org/lookup/id/{gene_id}?expand=0"
    headers = {"Content-Type": "application/json"}
    response = requests.get(url_lookup, headers=headers)
    if not response.ok:
        raise ValueError(f"Gene ID {gene_id} not found in Ensembl database.")
    
    data = response.json()
    chrom = data['seq_region_name']
    strand = data['strand']
    gene_name = data.get('display_name', gene_id)
    
    # Calculate the promoter window based on the DNA strand (+ or -)
    if strand == 1:
        tss = data['start']
        prom_start = tss - upstream
        prom_end = tss + downstream - 1
    else:
        tss = data['end']
        prom_start = tss - downstream + 1
        prom_end = tss + upstream
        
    print(f"Gene located ({gene_name}): Chromosome {chrom} | TSS: {tss} | Strand: {strand}")
    print(f"Downloading promoter sequence ({upstream} bp upstream, {downstream} bp downstream)...")
    
    url_seq = f"https://rest.ensembl.org/sequence/region/homo_sapiens/{chrom}:{prom_start}..{prom_end}:{strand}"
    headers_seq = {"Content-Type": "text/plain"}
    response_seq = requests.get(url_seq, headers=headers_seq)
    response_seq.raise_for_status()
    
    sequence = response_seq.text
    print(f"Sequence downloaded successfully. Length: {len(sequence)} bp.")
    return sequence, gene_name

In [3]:
# SETUP MODULE 2: MACHINE LEARNING MODEL & DATABASES (JASPAR / HOCOMOCO)

def generate_synthetic_dataset(n_samples=3000):
# Generates synthetic data simulating real vs. false positive TFBS contexts.
    np.random.seed(42)
    X, y = [], []
    classes = np.random.choice([0, 1], size=n_samples, p=[0.75, 0.25])
    
    for is_real in classes:
        if is_real == 1:
            score = np.clip(np.random.normal(88, 5), 75, 100)
            gc_motif = np.clip(np.random.normal(0.48, 0.08), 0, 1)
            gc_up = np.clip(np.random.normal(0.48, 0.08), 0, 1)
            gc_down = np.clip(np.random.normal(0.48, 0.08), 0, 1)
        else:
            score = np.clip(np.random.normal(84, 8), 75, 100)
            gc_base = np.random.normal(0.75, 0.1) if np.random.rand() > 0.5 else np.random.normal(0.25, 0.1)
            gc_motif = np.clip(np.random.normal(gc_base, 0.1), 0, 1)
            gc_up = np.random.normal(gc_base, 0.1)
            gc_down = np.random.normal(gc_base, 0.1)
            
        X.append([score, gc_motif, gc_up, gc_down])
        y.append(is_real)
        
    return pd.DataFrame(X, columns=['JASPAR_Score', 'GC_Motif', 'GC_Up', 'GC_Down']), pd.Series(y)

def train_ml_filter_model():
# Trains a Random Forest classifier to predict epigenetic viability.
    print("Machine Learning (Random Forest) to filter false positives")
    X, y = generate_synthetic_dataset(3000)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    rf = RandomForestClassifier(n_estimators=100, max_depth=5, min_samples_leaf=10, class_weight='balanced', random_state=42)
    rf.fit(X_train, y_train)
    return rf

def fetch_all_human_tfs():
#Iterates through JASPAR API pages to download the complete human TF catalog
    print("Connecting to JASPAR to fetch the COMPLETE human TF catalog")
    url = "https://jaspar.genereg.net/api/v1/matrix/?collection=CORE&tax_group=vertebrates&species=9606&page_size=1000"
    tfs = []
    
    while url:
        response = requests.get(url).json()
        tfs.extend([{'id': res['matrix_id'], 'name': res['name']} for res in response['results']])
        url = response.get('next') 
        
    print(f"Downloaded {len(tfs)} validated Transcription Factors.")
    return tfs

def download_hocomoco_matrix(tf_name):
# Fetches secondary validation matrices from HOCOMOCO v11
    clean_name = tf_name.split('::')[0]
    clean_name = re.sub(r'[^A-Za-z0-9]', '', clean_name).upper()
    variants = ['0.A', '0.B', '0.C', '1.A', '1.B']
    
    for var in variants:
        url = f"https://hocomoco11.autosome.org/final_bundle/hocomoco11/core/HUMAN/mono/pcm/{clean_name}_HUMAN.H11MO.{var}.pcm"
        response = requests.get(url)
        if response.ok:
            lines = response.text.strip().split('\n')
            counts = {'A': [], 'C': [], 'G': [], 'T': []}
            for line in lines:
                if line.startswith('>'): continue
                parts = [float(x) for x in line.split()]
                if len(parts) >= 4:
                    counts['A'].append(parts[0])
                    counts['C'].append(parts[1])
                    counts['G'].append(parts[2])
                    counts['T'].append(parts[3])
            if len(counts['A']) > 0:
                return motifs.Motif(counts=counts)
    return None

def calculate_gc(sequence):
# Calculates GC content percentage of a given sequence.
    if not sequence: return 0.5
    sec_upper = sequence.upper()
    return (sec_upper.count('G') + sec_upper.count('C')) / len(sequence)

def remove_overlaps_and_duplicates(df_results):
# Clustering algorithm to remove redundant overlapping binding sites
    if df_results.empty: return df_results
    df_results['H_Score_Num'] = pd.to_numeric(df_results['HOCOMOCO_Score(%)'].replace('Not in DB', 0))
    df = df_results.sort_values(by=['TF_Factor', 'JASPAR_Score(%)', 'H_Score_Num', 'AI_Confidence(%)'], ascending=[True, False, False, False])
    
    clean_rows = []
    seen = {}
    for _, row in df.iterrows():
        tf = row['TF_Factor']
        start, end = row['Start'], row['End']
        if tf not in seen:
            seen[tf] = [(start, end)]
            clean_rows.append(row.drop('H_Score_Num'))
        else:
            overlap = False
            for (v_start, v_end) in seen[tf]:
                if max(start, v_start) <= min(end, v_end) + 10:
                    overlap = True
                    break
            if not overlap:
                seen[tf].append((start, end))
                clean_rows.append(row.drop('H_Score_Num'))
                
    return pd.DataFrame(clean_rows)

In [4]:
# SETUP MODULE 3: ENRICHR API INTEGRATION

ENRICHR_URL = 'https://maayanlab.cloud/Enrichr'

def analyze_with_enrichr(unique_tfs_list):

# Submits the list of unique TFs to Enrichr and retrieves statistically significant KEGG and DisGeNET associations.
    
    print(f"Connecting to MaayanLab Enrichr... Submitting {len(unique_tfs_list)} unique factors.")
    
    genes_str = '\n'.join(unique_tfs_list)
    payload = {
        'list': (None, genes_str),
        'description': (None, 'TFs_Promoter_Analysis')
    }
    
    response = requests.post(f'{ENRICHR_URL}/addList', files=payload, verify=False)
    if not response.ok:
        raise Exception("Failed to communicate with Enrichr server.")
        
    user_list_id = response.json()['userListId']
    print("List accepted by Enrichr. Downloading enrichment data.")

    annotations = {gene: {'KEGG': [], 'Cancer': []} for gene in unique_tfs_list}

    # Query KEGG 2021 Human
    res_kegg = requests.get(f'{ENRICHR_URL}/enrich?userListId={user_list_id}&backgroundType=KEGG_2021_Human', verify=False).json()
    for row in res_kegg.get('KEGG_2021_Human', []):
        term = row[1]
        involved_genes = row[5]
        for gene in involved_genes:
            if gene in annotations:
                annotations[gene]['KEGG'].append(term)

    # Query DisGeNET
    res_disgenet = requests.get(f'{ENRICHR_URL}/enrich?userListId={user_list_id}&backgroundType=DisGeNET', verify=False).json()
    cancer_keywords = ['cancer', 'carcinoma', 'leukemia', 'melanoma', 'glioma', 'sarcoma', 'lymphoma', 'tumor', 'neoplasm', 'adenoma']
    
    for row in res_disgenet.get('DisGeNET', []):
        disease = row[1]
        involved_genes = row[5]
        if any(keyword in disease.lower() for keyword in cancer_keywords):
            for gene in involved_genes:
                if gene in annotations:
                    annotations[gene]['Cancer'].append(disease)

    return annotations

def map_enrichr_data(df_clean):
# Cross-references cleaned results with Enrichr API and generates filtered DataFrames
    print("\n Analyzing via Enrichr API")
    
    raw_tfs = df_clean['TF_Factor'].unique()
    clean_tfs = set()
    tf_mapping = {} 
    
    for tf in raw_tfs:
        base_tf = re.split(r'::|-', str(tf).upper())[0]
        clean_tfs.add(base_tf)
        tf_mapping[tf] = base_tf
        
    tf_query_list = list(clean_tfs)
    
    try:
        annotations = analyze_with_enrichr(tf_query_list)
    except Exception as e:
        print(f"ERROR: Enrichr mapping failed: {e}")
        return pd.DataFrame(), pd.DataFrame()

    def get_annotation(original_tf, category):
        base_tf = tf_mapping.get(original_tf, "")
        data = annotations.get(base_tf, {}).get(category, [])
        return " | ".join(data[:4]) if data else "No direct report"

    df_clean = df_clean.copy()
    df_clean['KEGG_Pathways'] = df_clean['TF_Factor'].apply(lambda x: get_annotation(x, 'KEGG'))
    df_clean['DisGeNET_Cancers'] = df_clean['TF_Factor'].apply(lambda x: get_annotation(x, 'Cancer'))

    filter_cancer = df_clean['DisGeNET_Cancers'] != "No direct report"
    df_cancer = df_clean[filter_cancer].copy()

    filter_pathways = (df_clean['DisGeNET_Cancers'] == "No direct report") & (df_clean['KEGG_Pathways'] != "No direct report")
    df_pathways = df_clean[filter_pathways].copy()

    return df_cancer, df_pathways

In [ ]:
# PIPELINE EXECUTION
  
if __name__ == "__main__":
    print(f"INITIATING PROMOTER SCAN FOR ID: {TARGET_GENE_ID}")

    try:
        # Create output directory if it doesn't exist
        if not os.path.exists(OUTPUT_DIR):
            os.makedirs(OUTPUT_DIR)

        promoter_sequence, gene_name = download_ensembl_promoter(TARGET_GENE_ID, upstream=UPSTREAM_BP, downstream=DOWNSTREAM_BP)
        ml_model = train_ml_filter_model()
        tf_list = fetch_all_human_tfs()
        
        successful_results = []
        
        print("\n Initiating Scan")
        
        for i, tf in enumerate(tf_list):
            if i % 100 == 0 and i > 0:
                print(f"   ... processed {i}/{len(tf_list)} factors...")
                
            try:
                url_matrix = f"https://jaspar.genereg.net/api/v1/matrix/{tf['id']}.jaspar"
                response = requests.get(url_matrix)
                if not response.ok: continue
                jaspar_motif = motifs.read(StringIO(response.text), "jaspar")
                jaspar_pwm = jaspar_motif.counts.normalize(pseudocounts=0.5).log_odds()
                
                max_score_j, min_score_j = jaspar_pwm.max, jaspar_pwm.min
                abs_threshold_j = JASPAR_THRESHOLD * (max_score_j - min_score_j) + min_score_j
                
                hocomoco_motif = download_hocomoco_matrix(tf['name'])
                
                # Main Sequence Scan
                for pos, score in jaspar_pwm.search(Seq(promoter_sequence), threshold=abs_threshold_j):
                    rel_score_jaspar = (score - min_score_j) / (max_score_j - min_score_j) * 100
                    strand = "+" if pos >= 0 else "-"
                    start = pos + 1 if pos >= 0 else pos + len(promoter_sequence) + 1
                    end = start + len(jaspar_motif) - 1
                    
                    # HOCOMOCO Confirmation Scan
                    rel_score_hocomoco = None
                    if hocomoco_motif:
                        hocomoco_pwm = hocomoco_motif.counts.normalize(pseudocounts=0.5).log_odds()
                        max_sh, min_sh = hocomoco_pwm.max, hocomoco_pwm.min
                        
                        idx_h_start = max(0, start - 10)
                        idx_h_end = min(len(promoter_sequence), end + 10)
                        window_h = Seq(promoter_sequence[idx_h_start:idx_h_end])
                        
                        best_h_scores = []
                        for pos_h, score_h in hocomoco_pwm.search(window_h, threshold=min_sh):
                            best_h_scores.append(score_h)
                            
                        if best_h_scores:
                            best_sc_h = max(best_h_scores)
                            rel_score_hocomoco = (best_sc_h - min_sh) / (max_sh - min_sh) * 100
                    
                    # ML Filter Application
                    idx_start = max(0, start - 1)
                    idx_end = min(len(promoter_sequence), end)
                    mot_seq = promoter_sequence[idx_start:idx_end]
                    up_seq = promoter_sequence[max(0, idx_start-20) : idx_start]
                    down_seq = promoter_sequence[idx_end : min(len(promoter_sequence), idx_end+20)]
                    
                    features = [calculate_gc(mot_seq), calculate_gc(up_seq), calculate_gc(down_seq)]
                    pred_data = pd.DataFrame([[rel_score_jaspar] + features], columns=['JASPAR_Score', 'GC_Motif', 'GC_Up', 'GC_Down'])
                    
                    probability = ml_model.predict_proba(pred_data)[0][1]
                    
                    if probability > 0.85:
                        successful_results.append({
                            'Gene': gene_name,
                            'TF_Factor': tf['name'],
                            'Strand': strand,
                            'Start': start,
                            'End': end,
                            'JASPAR_Score(%)': round(rel_score_jaspar, 2),
                            'HOCOMOCO_Score(%)': round(rel_score_hocomoco, 2) if rel_score_hocomoco else "Not in DB",
                            'AI_Confidence(%)': round(probability * 100, 2),
                            'Sequence': str(mot_seq)
                        })
                        
            except Exception:
                pass
                
        if successful_results:
            df_raw = pd.DataFrame(successful_results)
            df_clean = remove_overlaps_and_duplicates(df_raw)
            
            # Sort by JASPAR > HOCOMOCO > AI
            df_clean['H_Score_Num'] = pd.to_numeric(df_clean['HOCOMOCO_Score(%)'].replace('Not in DB', 0))
            df_clean = df_clean.sort_values(by=['JASPAR_Score(%)', 'H_Score_Num', 'AI_Confidence(%)'], ascending=[False, False, False])
            df_clean = df_clean.drop(columns=['H_Score_Num'])
            
            base_file_name = os.path.join(OUTPUT_DIR, f"Base_Results_{gene_name}.csv")
            df_clean.to_csv(base_file_name, index=False)
            
            # FILTERING (ENRICHR API)
            df_cancer, df_pathways = map_enrichr_data(df_clean)
            
            cancer_file_name = os.path.join(OUTPUT_DIR, f"Oncological_Filter_{gene_name}.csv")
            pathways_file_name = os.path.join(OUTPUT_DIR, f"Metabolic_Pathways_{gene_name}.csv")
            
            if not df_cancer.empty:
                df_cancer = df_cancer.sort_values(by=['AI_Confidence(%)', 'JASPAR_Score(%)'], ascending=[False, False])
                df_cancer.to_csv(cancer_file_name, index=False)
                
            if not df_pathways.empty:
                 df_pathways = df_pathways.sort_values(by=['AI_Confidence(%)', 'JASPAR_Score(%)'], ascending=[False, False])
                 df_pathways.to_csv(pathways_file_name, index=False)
            
            print("\n" + "="*75)
            print(f"PIPELINE COMPLETED!")
            print(f"1. Base Results (All TFs): {base_file_name}")
            
            if not df_cancer.empty:
                print(f"2. ONCOLOGICAL FILTER: {cancer_file_name}")
                print(f"   -> Isolated {len(df_cancer)} factors linked to tumors in Enrichr.")
            if not df_pathways.empty:
                 print(f"3. Metabolic Pathways Filter: {pathways_file_name}")
            
            print("="*75)
            
            if not df_cancer.empty:
                print("\n CONFIRMED TARGETS:")
                print(df_cancer[['TF_Factor', 'DisGeNET_Cancers', 'AI_Confidence(%)']].head(5).to_string(index=False))
        else:
            print("\n WARNING: Scan finished. No TFs found that surpassed the strict filters.")
            
    except Exception as e:
        print(f"\n ERROR: Execution failed: {e}")